# LangGraph Level 2A — State & Routing

Patterns covered:
1. **MessagesState** — preconfigured state with `add_messages` reducer
2. **Command** — update state + decide next node in a single object
3. **Routing Pattern** — LLM classifier that decides which branch to execute
4. **Prompt Chaining Pattern** — A→B→C with intermediate validation

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from typing import Literal, TypedDict, Annotated
import operator
from IPython.display import Image, display

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.types import Command

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

---
## 1. MessagesState

**What is it?** A preconfigured LangGraph state that already includes `messages: Annotated[list, add_messages]`.
No need to manually define a `TypedDict` or import `add_messages`.

**When to use it?** Whenever the graph primarily works with conversation (chat history).

```
# Without MessagesState (verbose):
class State(TypedDict):
    messages: Annotated[list, add_messages]

# With MessagesState (clean):
class State(MessagesState):
    extra_field: str   # only add what you need extra
```

In [ ]:
# MessagesState already has: messages: Annotated[list[AnyMessage], add_messages]
# It can be extended with extra fields
class AgentState(MessagesState):
    summary: str  # optional extra field

def chat_node(state: AgentState) -> AgentState:
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

builder = StateGraph(AgentState)
builder.add_node("chat", chat_node)
builder.add_edge(START, "chat")
builder.add_edge("chat", END)
graph = builder.compile()

# Invoke — the add_messages reducer appends automatically
result = graph.invoke({"messages": [HumanMessage(content="What is LangGraph in one sentence?")]})
print("Messages in state:", len(result["messages"]), "messages")
print("Response:", result["messages"][-1].content)

---
## 2. Command — update state and route in a single object

**What is it?** `Command` combines two operations that previously required separate code:
- `update`: update state fields
- `goto`: decide the next node

**Without Command** (previous approach):
```python
def node(state): return {"field": "value"}  # updates
def router(state): return "next_node"        # separate routing
graph.add_conditional_edges("node", router)
```

**With Command** (cleaner):
```python
def node(state) -> Command:
    return Command(update={"field": "value"}, goto="next_node")  # all together
```

In [ ]:
class PipelineState(MessagesState):
    step: str
    result: str

def classifier_node(state: PipelineState) -> Command[Literal["process_code", "process_text"]]:
    """Classifies the input and routes + updates state in a single Command."""
    last_msg = state["messages"][-1].content
    
    # Classify with LLM
    response = llm.invoke([
        SystemMessage(content="Reply ONLY with 'code' or 'text' depending on the type of question."),
        HumanMessage(content=last_msg)
    ])
    category = response.content.strip().lower()
    next_node = "process_code" if category == "code" else "process_text"
    
    # Command: updates state AND decides routing simultaneously
    return Command(
        update={"step": f"classified_as_{category}"},
        goto=next_node
    )

def process_code(state: PipelineState) -> PipelineState:
    response = llm.invoke(state["messages"] + [
        SystemMessage(content="You are a code expert. Reply in a technical and concise manner.")
    ])
    return {"result": "[CODE]", "messages": [response]}

def process_text(state: PipelineState) -> PipelineState:
    response = llm.invoke(state["messages"] + [
        SystemMessage(content="You are a communication expert. Reply in a clear and friendly manner.")
    ])
    return {"result": "[TEXT]", "messages": [response]}

builder = StateGraph(PipelineState)
builder.add_node("classifier", classifier_node)
builder.add_node("process_code", process_code)
builder.add_node("process_text", process_text)
builder.add_edge(START, "classifier")
builder.add_edge("process_code", END)
builder.add_edge("process_text", END)
graph_cmd = builder.compile()

display(Image(graph_cmd.get_graph().draw_mermaid_png()))

In [ ]:
# Test with a code question
result = graph_cmd.invoke({"messages": [HumanMessage(content="How do I write a loop in Python?")]})
print(f"Step: {result['step']}")
print(f"Response: {result['messages'][-1].content[:200]}")

print("---")

# Test with a text question
result2 = graph_cmd.invoke({"messages": [HumanMessage(content="How do I explain AI to my grandmother?")]})
print(f"Step: {result2['step']}")
print(f"Response: {result2['messages'][-1].content[:200]}")

---
## 3. Routing Pattern — LLM classifier → specialized branches

**What is it?** A classifier node that reads the input and decides which specialist to execute.
It is the core of any multi-agent system. In Level 3 this classifier is called a Supervisor.

```
Input → [Classifier] → "technical" → [TechExpert]
                     → "billing"   → [BillingAgent] → Output
                     → "general"  → [GeneralAgent]
```

In [ ]:
from pydantic import BaseModel

class RouteDecision(BaseModel):
    category: Literal["technical", "billing", "general"]
    reasoning: str

# LLM with structured output for reliable routing
router_llm = llm.with_structured_output(RouteDecision)

class SupportState(MessagesState):
    category: str
    agent_response: str

def route_request(state: SupportState) -> Command[Literal["technical_agent", "billing_agent", "general_agent"]]:
    """Classifier that routes support tickets to the correct agent."""
    last_msg = state["messages"][-1].content
    decision = router_llm.invoke([
        SystemMessage(content=(
            "Classify this support ticket:\n"
            "- technical: bugs, errors, technical issues\n"
            "- billing: payments, invoices, subscriptions\n"
            "- general: general questions, information"
        )),
        HumanMessage(content=last_msg)
    ])
    return Command(
        update={"category": decision.category},
        goto=f"{decision.category}_agent"
    )

def technical_agent(state: SupportState) -> SupportState:
    resp = llm.invoke([
        SystemMessage(content="You are technical support. Reply with a step-by-step technical solution."),
        *state["messages"]
    ])
    return {"agent_response": resp.content, "messages": [resp]}

def billing_agent(state: SupportState) -> SupportState:
    resp = llm.invoke([
        SystemMessage(content="You are billing support. Reply about payments and subscriptions."),
        *state["messages"]
    ])
    return {"agent_response": resp.content, "messages": [resp]}

def general_agent(state: SupportState) -> SupportState:
    resp = llm.invoke([
        SystemMessage(content="You are general support. Reply in a friendly and informative manner."),
        *state["messages"]
    ])
    return {"agent_response": resp.content, "messages": [resp]}

builder = StateGraph(SupportState)
builder.add_node("router", route_request)
builder.add_node("technical_agent", technical_agent)
builder.add_node("billing_agent", billing_agent)
builder.add_node("general_agent", general_agent)
builder.add_edge(START, "router")
builder.add_edge("technical_agent", END)
builder.add_edge("billing_agent", END)
builder.add_edge("general_agent", END)
support_graph = builder.compile()

display(Image(support_graph.get_graph().draw_mermaid_png()))

In [ ]:
tickets = [
    "My app crashes when I upload large files",
    "When will I receive this month's invoice?",
    "Can I use the platform from a mobile device?"
]

for ticket in tickets:
    result = support_graph.invoke({"messages": [HumanMessage(content=ticket)]})
    print(f"Ticket: {ticket}")
    print(f"  → Category: {result['category']}")
    print(f"  → Response: {result['agent_response'][:150]}...")
    print()

---
## 4. Prompt Chaining Pattern — A→B→C with intermediate validation

**What is it?** A linear pipeline where each node transforms the output of the previous one.
A validator node is added between steps that can redirect if the output does not meet criteria.

```
[research] → [validate] → OK → [draft] → [validate] → OK → [finalize]
                       → FAIL → [research] (retry)
```

**Difference from Routing:** Routing = input → one branch. Chaining = linear sequence with transformations.

In [ ]:
from pydantic import BaseModel as PydanticBaseModel

class ValidationResult(PydanticBaseModel):
    is_valid: bool
    feedback: str

validator_llm = llm.with_structured_output(ValidationResult)

class BlogState(MessagesState):
    topic: str
    research: str
    draft: str
    final_post: str
    validation_feedback: str
    retry_count: int

def research_node(state: BlogState) -> BlogState:
    """Node A: researches the topic."""
    response = llm.invoke([
        SystemMessage(content="You are a researcher. Generate 3 key points about the given topic."),
        HumanMessage(content=f"Topic: {state['topic']}")
    ])
    return {"research": response.content, "retry_count": state.get("retry_count", 0)}

def validate_research(state: BlogState) -> Command[Literal["draft_node", "research_node"]]:
    """Intermediate validator: is the research sufficient to write the post?"""
    result = validator_llm.invoke([
        SystemMessage(content="Evaluate whether this research has enough content for a blog post (minimum 3 concrete points)."),
        HumanMessage(content=state["research"])
    ])
    
    retry = state.get("retry_count", 0)
    if result.is_valid or retry >= 2:  # max 2 retries
        return Command(
            update={"validation_feedback": result.feedback},
            goto="draft_node"
        )
    return Command(
        update={"validation_feedback": result.feedback, "retry_count": retry + 1},
        goto="research_node"  # retry
    )

def draft_node(state: BlogState) -> BlogState:
    """Node B: writes the post draft using the research."""
    response = llm.invoke([
        SystemMessage(content="You are a blog writer. Write a ~200 word post based on the research."),
        HumanMessage(content=f"Research:\n{state['research']}\n\nTopic: {state['topic']}")
    ])
    return {"draft": response.content}

def finalize_node(state: BlogState) -> BlogState:
    """Node C: polishes the final draft with a title and formatting."""
    response = llm.invoke([
        SystemMessage(content="Add an engaging title and format the post with clear sections."),
        HumanMessage(content=state["draft"])
    ])
    return {"final_post": response.content}

builder = StateGraph(BlogState)
builder.add_node("research_node", research_node)
builder.add_node("validate_research", validate_research)
builder.add_node("draft_node", draft_node)
builder.add_node("finalize_node", finalize_node)

builder.add_edge(START, "research_node")
builder.add_edge("research_node", "validate_research")
# validate_research uses Command to decide → draft_node or → research_node
builder.add_edge("draft_node", "finalize_node")
builder.add_edge("finalize_node", END)

chain_graph = builder.compile()
display(Image(chain_graph.get_graph().draw_mermaid_png()))

In [ ]:
result = chain_graph.invoke({"topic": "Why LangGraph outperforms prompt-based agents"})

print("=== RESEARCH ===")
print(result["research"][:300])
print("\n=== VALIDATION ===")
print(result["validation_feedback"])
print(f"Retries: {result['retry_count']}")
print("\n=== FINAL POST ===")
print(result["final_post"][:500])

---
## Takeaways — N2A

| Pattern | When to use it | Key point |
|---------|---------------|-----------|
| `MessagesState` | Whenever you work with chat history | Already includes `add_messages` reducer |
| `Command` | When a node must update state AND decide the next step | Replaces the pair (return dict + conditional_edge) |
| Routing | Classify input → specialists | LLM with structured output for reliable routing |
| Prompt Chaining | Pipeline A→B→C | Add a validator node between steps for quality |

**Next:** `N2B_hitl_streaming_memory.ipynb` — interrupt_before, per-node streaming, SQLite checkpointer